In [37]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pendulum as pdl
import datetime as dt

In [38]:

# Create a dataframe which defines the 
loadcase_timesheet = pd.DataFrame(columns=["LC","lc-title","timestamp"])
start_date = pd.to_datetime("2023-11-01 00:00:00")

# create the 10 loadsteps for the creep load case
simulation_duration = 365 #days
simulation_steps = 30
for i in range(1,simulation_steps+1):
    loadcase_timesheet.loc[i - 1] = {
        "LC": 40074 + i,
        "lc-title": "Creep until start of load test",
        "timestamp": start_date + i * pdl.Duration(
            days=simulation_duration / simulation_steps
        ),
    }

print(loadcase_timesheet.timestamp.dtype)
loadcase_timesheet.head(11)

datetime64[us]


,LC,lc-title,timestamp
0,40075,Creep until start of load test,2023-11-13 04:00:00
1,40076,Creep until start of load test,2023-11-25 08:00:00
2,40077,Creep until start of load test,2023-12-07 12:00:00
3,40078,Creep until start of load test,2023-12-19 16:00:00
4,40079,Creep until start of load test,2023-12-31 20:00:00
5,40080,Creep until start of load test,2024-01-13 00:00:00
6,40081,Creep until start of load test,2024-01-25 04:00:00
7,40082,Creep until start of load test,2024-02-06 08:00:00
8,40083,Creep until start of load test,2024-02-18 12:00:00
9,40084,Creep until start of load test,2024-03-01 16:00:00


In [39]:
# the sofistic produces CSV files with:
#  LC, Node, u-X, u-Y, u-Z, r-X, r-Y, r-Z, ...
# loaded_data = pd.read_csv("test-openlab-export.csv",)
loaded_data = pd.read_csv("openlab-sensors-export.csv",)
# del the first 2/3 of records
len(loaded_data)//3*2
loaded_data = loaded_data.iloc[len(loaded_data)//3*2:, :]
len(loaded_data)

446

In [40]:


sensors = pd.read_csv("Sensor_Names_autoextracted_from_model.csv",)
# loaded data format correction|
p = loaded_data.columns
loaded_data[p[3:]] = loaded_data[p[3:]].apply(lambda x: x.str.replace(',', '.'))
loaded_data[p[3:]] = loaded_data[p[3:]].apply(lambda x: pd.to_numeric(x, errors='coerce'))


def from_mrad_to_mm_per_m(mrad):
    """Convert mrad to mm/m"""
    # mrad.str.replace(",", ".", inplace=True)
    values = pd.to_numeric(mrad, errors="raise")
    return np.tan(values * 1e-3) * 1000

for _, sensor in sensors.iterrows():
    # get senor ref name
    refnum = sensor["sofistik_ref_number"]
    if refnum is None:
        continue
    # extract the relevant predictions
    dftemp = loaded_data["NODE "] == refnum
    dftemp = loaded_data[dftemp]
    if dftemp.empty:
        continue
    else:
        dftemp = dftemp.query("`LC-title ` == 'Creep until start of load test'")
        dftemp = dftemp.copy()
        QoI_dict = {
            "Tilt-X": from_mrad_to_mm_per_m(dftemp["phi-Y [mrad]"].values),
            "Tilt-Y": from_mrad_to_mm_per_m(dftemp["phi-X [mrad]"].values),
            "Displacement-X": dftemp["u-X [mm]"].values,
            "Displacement-Y": dftemp["u-Y [mm]"].values,
            "Displacement-Z": dftemp["u-Z [mm]"].values,
        }
        SensorNameDict = {
            "NGS": "TIL",
            "FT": "PE",
            "BU": "CB",
        }
        sensor_name = sensor["Name"]
        for key, value in SensorNameDict.items():
            sensor_name = sensor_name.replace(key, value)
        # create a new column with the sensor name
        try:
            qoi = QoI_dict[sensor["QoI"]]
        except KeyError:
            print(f"QoI {sensor['QoI']} not found for sensor {sensor['Name']}")
            continue
        loadcase_timesheet[sensor_name] = qoi
        
loadcase_timesheet.head(11)


,LC,lc-title,timestamp,G_TILX_PE11_CB1100_0,G_TILY_PE11_CB1100_0,G_TILX_PE12_CB1100_0,G_TILY_PE12_CB1100_0,G_TILX_PE13_CB1100_0,G_TILY_PE13_CB1100_0,G_TILX_PE21_CB0400_0,G_TILY_PE21_CB0400_0,G_TILX_PE22_CB0400_0,G_TILY_PE22_CB0400_0,G_TILX_PE23_CB0400_0,G_TILY_PE23_CB0400_0
0,40075,Creep until start of load test,2023-11-13 04:00:00,-1.376001,0.061,-0.462,0.009,-0.309,-0.044,-0.316,0.021,-0.317,-0.024,-0.328,-0.068
1,40076,Creep until start of load test,2023-11-25 08:00:00,-1.363001,0.061,-0.449,0.009,-0.296,-0.043,-0.342,0.020,-0.344,-0.025,-0.354,-0.068
2,40077,Creep until start of load test,2023-12-07 12:00:00,-1.353001,0.062,-0.440,0.009,-0.287,-0.043,-0.363,0.020,-0.365,-0.026,-0.375,-0.069
3,40078,Creep until start of load test,2023-12-19 16:00:00,-1.344001,0.062,-0.432,0.009,-0.279,-0.043,-0.383,0.019,-0.385,-0.027,-0.395,-0.070
4,40079,Creep until start of load test,2023-12-31 20:00:00,-1.337001,0.062,-0.426,0.009,-0.273,-0.043,-0.400,0.018,-0.401,-0.028,-0.412,-0.071
5,40080,Creep until start of load test,2024-01-13 00:00:00,-1.331001,0.062,-0.420,0.010,-0.268,-0.043,-0.415,0.017,-0.416,-0.029,-0.427,-0.072
6,40081,Creep until start of load test,2024-01-25 04:00:00,-1.325001,0.062,-0.415,0.010,-0.263,-0.043,-0.430,0.016,-0.431,-0.030,-0.441,-0.073
7,40082,Creep until start of load test,2024-02-06 08:00:00,-1.320001,0.062,-0.410,0.010,-0.259,-0.043,-0.443,0.015,-0.444,-0.031,-0.454,-0.073
8,40083,Creep until start of load test,2024-02-18 12:00:00,-1.316001,0.062,-0.406,0.010,-0.255,-0.043,-0.454,0.014,-0.455,-0.031,-0.466,-0.074
9,40084,Creep until start of load test,2024-03-01 16:00:00,-1.312001,0.062,-0.403,0.010,-0.251,-0.043,-0.465,0.014,-0.466,-0.032,-0.476,-0.075


In [41]:
# prform a join between the two dataframes to map the LC to timestamps 
# loaded_data.join(loadcase_timesheet.set_index("LC"), on=0, how="left", rsuffix="_timestamp")
        
loaded_data[loaded_data["LC-title "]=="Creep until start of load test" ].head()
cpdf = loaded_data[loaded_data["LC-title "]=="Creep until start of load test" ].copy()

cpdf["LC "] = cpdf["LC "].astype(str).str.strip()
loadcase_timesheet["LC"] = loadcase_timesheet["LC"].astype(str).str.strip()

timestamp_map = loadcase_timesheet.set_index("LC")["timestamp"]

print(timestamp_map.dtype)

# cpdf["timestamp"] = cpdf["LC "].map(timestamp_map)
cpdf["timestamp"] = cpdf["LC "].map(loadcase_timesheet.set_index("LC")["timestamp"])
cpdf.head()

datetime64[us]


,LC,LC-title,NODE,u-X [mm],u-Y [mm],u-Z [mm],phi-X [mrad],phi-Y [mrad],phi-Z [mrad],timestamp
1068,140075,Creep until start of load test,1140,2.248,0.006,-7.397,0.061,-1.376,0.000,NaT
1069,140075,Creep until start of load test,1250,-4.035,0.003,0.550,0.021,-0.316,-0.001,NaT
1070,140075,Creep until start of load test,2140,1.091,0.003,-2.651,0.009,-0.462,0.000,NaT
1071,140075,Creep until start of load test,2250,-3.603,-0.001,0.262,-0.024,-0.317,0.000,NaT
1072,140075,Creep until start of load test,3140,0.990,-0.001,-1.867,-0.044,-0.309,-0.001,NaT


In [42]:
loadcase_timesheet.head(5)
loadcase_timesheet.to_csv("loadcase_timesheet.csv", index=False)

In [43]:
print(cpdf.columns)
cpdf["timestamp"].head()


Index(['LC ', 'LC-title ', 'NODE ', 'u-X [mm]', 'u-Y [mm]', 'u-Z [mm]',
       'phi-X [mrad]', 'phi-Y [mrad]', 'phi-Z [mrad]', 'timestamp'],
      dtype='str')


1068   NaT
1069   NaT
1070   NaT
1071   NaT
1072   NaT
Name: timestamp, dtype: datetime64[us]

In [44]:
# cpdf.plot(x="LC ", y=['u-X [mm]', 'u-Y [mm]', 'u-Z [mm]'],
#           title="Displacement over time for Creep Load Case",
#           xlabel="Load Case", ylabel="Displacement [mm]", marker='o', linestyle='--', figsize=(10, 4))

# rotation_columns = ["phi-X [mrad]", "phi-Y [mrad]", "phi-Z [mrad]"]
# # .assign(y_scaled=lambda d: d[rotation_columns[2]] * 10).

# cpdf.plot(x="LC ", y=rotation_columns,
#           title="Displacement over time for Creep Load Case",
#           xlabel="Load Case", ylabel="Rotation [mrad*10]", marker='o', linestyle='--', figsize=(10, 4))

In [45]:
cpdf.columns
loadcase_timesheet.head(5)


,LC,lc-title,timestamp,G_TILX_PE11_CB1100_0,G_TILY_PE11_CB1100_0,G_TILX_PE12_CB1100_0,G_TILY_PE12_CB1100_0,G_TILX_PE13_CB1100_0,G_TILY_PE13_CB1100_0,G_TILX_PE21_CB0400_0,G_TILY_PE21_CB0400_0,G_TILX_PE22_CB0400_0,G_TILY_PE22_CB0400_0,G_TILX_PE23_CB0400_0,G_TILY_PE23_CB0400_0
0,40075,Creep until start of load test,2023-11-13 04:00:00,-1.376001,0.061,-0.462,0.009,-0.309,-0.044,-0.316,0.021,-0.317,-0.024,-0.328,-0.068
1,40076,Creep until start of load test,2023-11-25 08:00:00,-1.363001,0.061,-0.449,0.009,-0.296,-0.043,-0.342,0.020,-0.344,-0.025,-0.354,-0.068
2,40077,Creep until start of load test,2023-12-07 12:00:00,-1.353001,0.062,-0.440,0.009,-0.287,-0.043,-0.363,0.020,-0.365,-0.026,-0.375,-0.069
3,40078,Creep until start of load test,2023-12-19 16:00:00,-1.344001,0.062,-0.432,0.009,-0.279,-0.043,-0.383,0.019,-0.385,-0.027,-0.395,-0.070
4,40079,Creep until start of load test,2023-12-31 20:00:00,-1.337001,0.062,-0.426,0.009,-0.273,-0.043,-0.400,0.018,-0.401,-0.028,-0.412,-0.071


In [46]:
# save the csv dataframe
import os
os.remove("tiltmeters_prediction.csv")
loadcase_timesheet.to_csv("tiltmeters_prediction.csv", index=False)

In [47]:
loadcase_timesheet.columns

Index(['LC', 'lc-title', 'timestamp', 'G_TILX_PE11_CB1100_0',
       'G_TILY_PE11_CB1100_0', 'G_TILX_PE12_CB1100_0', 'G_TILY_PE12_CB1100_0',
       'G_TILX_PE13_CB1100_0', 'G_TILY_PE13_CB1100_0', 'G_TILX_PE21_CB0400_0',
       'G_TILY_PE21_CB0400_0', 'G_TILX_PE22_CB0400_0', 'G_TILY_PE22_CB0400_0',
       'G_TILX_PE23_CB0400_0', 'G_TILY_PE23_CB0400_0'],
      dtype='str')